In [8]:
"""
Steam 개인 맞춤 게임 추천 - XGBoost

사용 데이터
1. steam_user_games_merged.csv
   - steamid
   - game_name
   - playtime_hours
   - genre
   - tags

2. steam_top500_games.csv
   - appid
   - game_name
   - rating
   - genre
   - tags

3. Steam Web API
   - 사용자의 현재 전체 Steam 라이브러리 실시간 조회
   - GetOwnedGames API 사용

처리 과정

사용자 게임 CSV
    ↓
플레이타임 기반 가중치
    ↓
사용자 장르 선호도
    ↓
게임 Feature Vector
    ↓
학습 데이터 생성
    ↓
XGBoost
    ↓
좋아할 확률 예측
    ↓
Steam API에서 현재 라이브러리 조회
    ↓
이미 보유한 게임 제거
    ↓
추천 TOP N
"""


# ============================================================
# 라이브러리
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
import re
import warnings
import requests

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 0. 파일 설정
# ============================================================

USER_CSV = str(Path(__file__).resolve().parents[2] / "data" / "processed" / "steam_user_games_classified.csv") if "__file__" in dir() else "../../data/processed/steam_user_games_classified.csv"

TOP500_CSV = "../../data/processed/steam_top500_games_classified.csv"

OUTPUT_CSV = "../../reports/steam_xgboost_recommendations.csv"


# ============================================================
# 1. Steam API 설정
# ============================================================

STEAM_API_KEY = os.getenv("STEAM_API_KEY", "")


# ============================================================
# 2. 컬럼 설정
# ============================================================

COL_STEAM_ID = "steamid"
COL_GAME = "game_name"
COL_PLAYTIME = "playtime_hours"
COL_GENRE = "genre"
COL_TAGS = "tags"


# ============================================================
# 3. 장르 분류 기준
# ============================================================

GENRE_KEYWORDS = {

    "Action": [
        "action",
        "hack and slash",
        "beat 'em up",
        "beat em up",
        "fighting",
        "martial arts",
        "character action"
    ],

    "Adventure": [
        "adventure",
        "point & click",
        "point and click",
        "exploration",
        "walking simulator",
        "story rich",
        "interactive fiction"
    ],

    "RPG": [
        "rpg",
        "action rpg",
        "action-rpg",
        "jrpg",
        "crpg",
        "souls-like",
        "soulslike",
        "party-based rpg",
        "turn-based rpg",
        "turn based rpg",
        "old school rpg",
        "roguelike rpg",
        "role-playing"
    ],

    "Strategy": [
        "strategy",
        "real-time strategy",
        "real time strategy",
        "rts",
        "turn-based strategy",
        "turn based strategy",
        "4x",
        "grand strategy",
        "tactics",
        "tactical",
        "tower defense",
        "tower defence"
    ],

    "Simulation": [
        "simulation",
        "sim",
        "life sim",
        "farming sim",
        "city builder",
        "management",
        "building",
        "automation",
        "sandbox"
    ],

    "Shooter": [
        "shooter",
        "fps",
        "first-person shooter",
        "first person shooter",
        "third-person shooter",
        "third person shooter",
        "tps",
        "tactical shooter",
        "hero shooter",
        "looter shooter",
        "bullet hell"
    ],

    "Sports": [
        "sports",
        "football",
        "soccer",
        "basketball",
        "baseball",
        "tennis",
        "golf",
        "hockey",
        "volleyball",
        "wrestling"
    ],

    "Racing": [
        "racing",
        "racer",
        "driving",
        "automobile sim",
        "car"
    ],

    "Horror": [
        "horror",
        "survival horror",
        "psychological horror"
    ],

    "Survival": [
        "survival",
        "survival crafting",
        "open world survival craft",
        "crafting",
        "base building"
    ],

    "Platformer": [
        "platformer",
        "2d platformer",
        "3d platformer",
        "metroidvania"
    ],

    "Puzzle": [
        "puzzle",
        "logic",
        "match 3",
        "match-3"
    ],

    "Casual": [
        "casual",
        "relaxing",
        "family friendly"
    ]
}


GENRES = list(GENRE_KEYWORDS.keys())


# ============================================================
# 4. 문자열 처리
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def split_values(value):

    if value is None:
        return []

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    if value.lower() == "unknown":
        return []

    result = []

    for item in value.split(","):

        item = item.strip()

        if item:
            result.append(item)

    return result


# ============================================================
# 5. 장르 추출
# ============================================================

def detect_genres(values):

    detected = []

    for value in values:

        value = normalize_text(value)

        if not value:
            continue

        for genre, keywords in GENRE_KEYWORDS.items():

            for keyword in keywords:

                keyword = normalize_text(keyword)

                if (
                    value == keyword
                    or keyword in value
                ):

                    if genre not in detected:
                        detected.append(genre)

                    break

    return detected


def determine_genres(genre, tags):

    # --------------------------------------------------------
    # 1. Steam Genre 우선
    # --------------------------------------------------------

    genre_values = split_values(genre)

    detected = detect_genres(
        genre_values
    )

    if detected:
        return detected


    # --------------------------------------------------------
    # 2. Genre가 없으면 Tags 사용
    # --------------------------------------------------------

    tag_values = split_values(tags)

    detected = detect_genres(
        tag_values
    )

    if detected:
        return detected


    # --------------------------------------------------------
    # 3. 둘 다 없으면 Unknown
    # --------------------------------------------------------

    return ["Unknown"]


# ============================================================
# 6. 게임을 Feature Vector로 변환
# ============================================================

def make_game_vector(
    genre,
    tags
):

    detected_genres = determine_genres(
        genre,
        tags
    )

    vector = {}

    for g in GENRES:

        if g in detected_genres:
            vector[g] = 1
        else:
            vector[g] = 0

    return vector


# ============================================================
# 7. 사용자 데이터 및 TOP500 데이터 로드
# ============================================================

def load_data():

    print("=" * 70)
    print("CSV 파일 로딩")
    print("=" * 70)

    user_df = pd.read_csv(
        USER_CSV,
        dtype={
            COL_STEAM_ID: str
        }
    )

    top500_df = pd.read_csv(
        TOP500_CSV,
        dtype={
            "appid": str
        }
    )

    print(
        f"사용자 데이터: {len(user_df):,}개"
    )

    print(
        f"TOP500 데이터: {len(top500_df):,}개"
    )

    return user_df, top500_df


# ============================================================
# 8. 사용자별 데이터 추출
#
# 이 부분은 선호도 학습용입니다.
# 추천에서 보유 게임을 제외하는 용도로는 사용하지 않습니다.
# ============================================================

def get_user_games(
    user_df,
    steam_id
):

    steam_id = str(
        steam_id
    ).strip()

    df = user_df[
        user_df[
            COL_STEAM_ID
        ].astype(str).str.strip()
        == steam_id
    ].copy()

    if df.empty:

        raise ValueError(
            f"Steam ID '{steam_id}'의 "
            "데이터를 찾을 수 없습니다."
        )


    # --------------------------------------------------------
    # 플레이타임 숫자 변환
    # --------------------------------------------------------

    df[
        COL_PLAYTIME
    ] = pd.to_numeric(
        df[COL_PLAYTIME],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            COL_PLAYTIME
        ]
    )


    # --------------------------------------------------------
    # 플레이타임 > 0
    # --------------------------------------------------------

    df = df[
        df[COL_PLAYTIME] > 0
    ].copy()


    # --------------------------------------------------------
    # 게임별 중복 제거
    # --------------------------------------------------------

    df = (
        df.groupby(
            [
                COL_GAME,
                COL_GENRE,
                COL_TAGS
            ],
            dropna=False,
            as_index=False
        )[COL_PLAYTIME]
        .sum()
    )


    return df


# ============================================================
# 9. Steam API에서 사용자의 전체 라이브러리 가져오기
#
# ★★★ 새롭게 추가된 핵심 부분 ★★★
#
# CSV가 아니라 Steam API를 실시간으로 조회합니다.
# ============================================================

def get_owned_games_from_steam(
    steam_id
):

    print()
    print("=" * 70)
    print("Steam 라이브러리 실시간 조회")
    print("=" * 70)


    # --------------------------------------------------------
    # API Key 확인
    # --------------------------------------------------------

    if (
        not STEAM_API_KEY
        or STEAM_API_KEY
        == "여기에_본인의_Steam_API_KEY"
    ):

        raise ValueError(
            "STEAM_API_KEY에 "
            "본인의 Steam Web API Key를 입력해주세요."
        )


    url = (
        "https://api.steampowered.com/"
        "IPlayerService/GetOwnedGames/v1/"
    )


    params = {

        "key":
            STEAM_API_KEY,

        "steamid":
            steam_id,

        "include_appinfo":
            True,

        "include_played_free_games":
            True

    }


    try:

        response = requests.get(

            url,

            params=params,

            timeout=10

        )

        response.raise_for_status()

        data = response.json()


    except requests.RequestException as e:

        raise ValueError(
            f"Steam API 요청 실패: {e}"
        )


    # --------------------------------------------------------
    # API response
    # --------------------------------------------------------

    response_data = data.get(
        "response",
        {}
    )


    games = response_data.get(
        "games",
        []
    )


    # --------------------------------------------------------
    # Steam API에서 게임 목록을 못 받은 경우
    # --------------------------------------------------------

    if not games:

        raise ValueError(

            "Steam 라이브러리를 가져오지 못했습니다.\n\n"

            "가능한 원인:\n"

            "1. Steam 프로필이 비공개입니다.\n"

            "2. 게임 세부 정보가 비공개입니다.\n"

            "3. Steam ID가 잘못되었습니다.\n"

            "4. Steam API Key가 잘못되었습니다.\n"

            "5. Steam API에서 일시적으로 데이터를 가져오지 못했습니다."

        )


    # --------------------------------------------------------
    # AppID 추출
    # --------------------------------------------------------

    owned_appids = set()


    for game in games:

        appid = game.get(
            "appid"
        )

        if appid is not None:

            owned_appids.add(
                str(appid).strip()
            )


    print(
        f"현재 Steam 라이브러리: "
        f"{len(owned_appids):,}개"
    )


    return owned_appids


# ============================================================
# 10. 플레이타임 가중치
# ============================================================

def calculate_weights(
    user_games
):

    df = user_games.copy()

    playtime = (
        df[COL_PLAYTIME]
        .astype(float)
        .to_numpy()
    )

    log_playtime = np.log1p(
        playtime
    )

    total = log_playtime.sum()

    if total == 0:

        df["weight"] = (
            1 / len(df)
        )

    else:

        df["weight"] = (
            log_playtime
            / total
        )

    return df


# ============================================================
# 11. User Preference Vector
# ============================================================

def build_user_preference(
    weighted_games
):

    preference = {
        genre: 0.0
        for genre in GENRES
    }


    for _, row in weighted_games.iterrows():

        genres = determine_genres(

            row[COL_GENRE],

            row[COL_TAGS]

        )

        weight = float(
            row["weight"]
        )


        for genre in genres:

            if genre in preference:

                preference[genre] += weight


    # --------------------------------------------------------
    # 다시 합이 1이 되도록 정규화
    # --------------------------------------------------------

    total = sum(
        preference.values()
    )

    if total > 0:

        for genre in preference:

            preference[genre] /= total


    return preference


# ============================================================
# 12. 게임 Feature 생성
# ============================================================

def create_game_features(
    top500_df
):

    rows = []


    for _, row in top500_df.iterrows():

        vector = make_game_vector(

            row.get(
                COL_GENRE,
                ""
            ),

            row.get(
                COL_TAGS,
                ""
            )

        )


        result = {

            "appid":
                str(row["appid"]).strip(),

            "game_name":
                row[COL_GAME]

        }


        # ----------------------------------------------------
        # 장르 Feature
        # ----------------------------------------------------

        for genre in GENRES:

            result[
                f"game_{genre}"
            ] = vector[genre]


        # ----------------------------------------------------
        # 평점
        # ----------------------------------------------------

        rating = pd.to_numeric(

            row.get(
                "rating",
                np.nan
            ),

            errors="coerce"

        )


        if pd.isna(rating):

            rating = 0


        result[
            "rating"
        ] = float(rating)


        rows.append(
            result
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 13. 학습 데이터 생성
#
# Positive:
# 사용자가 실제로 플레이한 게임
#
# Negative:
# 사용자가 플레이하지 않은 TOP500 게임
#
# 주의:
# 여기서는 기존 CSV를 이용합니다.
# Steam API의 라이브러리는 추천 제외용입니다.
# ============================================================

def create_training_data(
    user_games,
    game_features
):

    played_games = set(

        user_games[
            COL_GAME
        ]
        .astype(str)
        .str.lower()
        .str.strip()

    )


    X = []

    y = []


    # --------------------------------------------------------
    # 플레이한 게임
    # --------------------------------------------------------

    for _, game in game_features.iterrows():

        game_name = str(
            game["game_name"]
        ).lower().strip()


        feature = []


        for genre in GENRES:

            feature.append(

                game[
                    f"game_{genre}"
                ]

            )


        feature.append(
            game["rating"]
        )


        if game_name in played_games:

            X.append(
                feature
            )

            y.append(1)


    # --------------------------------------------------------
    # 플레이하지 않은 게임
    # --------------------------------------------------------

    for _, game in game_features.iterrows():

        game_name = str(
            game["game_name"]
        ).lower().strip()


        if game_name in played_games:

            continue


        feature = []


        for genre in GENRES:

            feature.append(

                game[
                    f"game_{genre}"
                ]

            )


        feature.append(
            game["rating"]
        )


        X.append(
            feature
        )

        y.append(0)


    X = np.array(
        X,
        dtype=float
    )

    y = np.array(
        y,
        dtype=int
    )


    return X, y


# ============================================================
# 14. XGBoost 학습
# ============================================================

def train_xgboost(
    X,
    y
):

    print()
    print("=" * 70)
    print("XGBoost 학습")
    print("=" * 70)


    print(
        f"학습 데이터: {len(X):,}개"
    )

    print(
        f"Positive: {sum(y == 1):,}개"
    )

    print(
        f"Negative: {sum(y == 0):,}개"
    )


    # --------------------------------------------------------
    # 데이터 분할
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = (
        train_test_split(

            X,

            y,

            test_size=0.2,

            random_state=42,

            stratify=y

        )
    )


    # --------------------------------------------------------
    # 클래스 불균형 보정
    # --------------------------------------------------------

    positive = np.sum(
        y_train == 1
    )

    negative = np.sum(
        y_train == 0
    )


    if positive > 0:

        scale_pos_weight = (
            negative
            / positive
        )

    else:

        scale_pos_weight = 1


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    model = XGBClassifier(

        n_estimators=300,

        max_depth=4,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        objective="binary:logistic",

        eval_metric="logloss",

        scale_pos_weight=scale_pos_weight,

        random_state=42,

        n_jobs=-1

    )


    model.fit(

        X_train,

        y_train

    )


    # --------------------------------------------------------
    # 테스트 예측
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test
    )

    y_prob = model.predict_proba(
        X_test
    )[:, 1]


    # --------------------------------------------------------
    # 평가
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )


    try:

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

    except:

        roc_auc = 0


    try:

        pr_auc = average_precision_score(
            y_test,
            y_prob
        )

    except:

        pr_auc = 0


    print()
    print("=== XGBoost 평가 ===")

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1-score : {f1:.4f}"
    )

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC   : {pr_auc:.4f}"
    )


    print()
    print("=== Classification Report ===")

    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )


    return model


# ============================================================
# 15. 추천 게임 계산
#
# ★ 이미 보유한 게임은 Steam API의 AppID를 기준으로 제외
# ============================================================

def recommend_games(
    model,
    game_features,
    user_preference,
    owned_appids,
    top_n=20
):

    df = game_features.copy()


    # --------------------------------------------------------
    # Feature 생성
    # --------------------------------------------------------

    X_recommend = []


    for _, game in df.iterrows():

        feature = []


        # ----------------------------------------------
        # 게임 장르 Feature
        # ----------------------------------------------

        for genre in GENRES:

            feature.append(

                game[
                    f"game_{genre}"
                ]

            )


        # ----------------------------------------------
        # 평점
        # ----------------------------------------------

        feature.append(
            game["rating"]
        )


        X_recommend.append(
            feature
        )


    X_recommend = np.array(
        X_recommend,
        dtype=float
    )


    # --------------------------------------------------------
    # XGBoost 확률
    # --------------------------------------------------------

    probabilities = (
        model
        .predict_proba(
            X_recommend
        )[:, 1]
    )


    df[
        "xgboost_probability"
    ] = probabilities


    # --------------------------------------------------------
    # 사용자 선호도와 게임 장르의 유사도
    # --------------------------------------------------------

    preference_scores = []


    for _, game in df.iterrows():

        score = 0.0


        for genre in GENRES:

            game_value = game[
                f"game_{genre}"
            ]

            user_value = user_preference[
                genre
            ]

            score += (
                game_value
                * user_value
            )


        preference_scores.append(
            score
        )


    df[
        "genre_preference_score"
    ] = preference_scores


    # --------------------------------------------------------
    # 최종 점수
    #
    # 현재는 XGBoost 확률을 최종 점수로 사용
    # --------------------------------------------------------

    df[
        "final_score"
    ] = df[
        "xgboost_probability"
    ]


    # ========================================================
    # ★ 핵심
    # Steam API에서 가져온 현재 라이브러리의 AppID 제거
    # ========================================================

    df[
        "appid"
    ] = (
        df["appid"]
        .astype(str)
        .str.strip()
    )


    owned_appids = set(
        str(appid).strip()
        for appid in owned_appids
    )


    before_count = len(df)


    df = df[
        ~df["appid"].isin(
            owned_appids
        )
    ].copy()


    after_count = len(df)


    removed_count = (
        before_count
        - after_count
    )


    print()
    print("=" * 70)
    print("이미 보유한 게임 제외")
    print("=" * 70)

    print(
        f"추천 후보 게임: "
        f"{before_count:,}개"
    )

    print(
        f"보유 게임으로 제외: "
        f"{removed_count:,}개"
    )

    print(
        f"최종 추천 후보: "
        f"{after_count:,}개"
    )


    # --------------------------------------------------------
    # 추천 순위
    # --------------------------------------------------------

    df = (
        df
        .sort_values(
            "final_score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


    df[
        "rank"
    ] = np.arange(
        1,
        len(df) + 1
    )


    return df


# ============================================================
# 16. 메인
# ============================================================

def main():

    print("=" * 70)

    print(
        "Steam XGBoost 개인 맞춤 추천 시스템"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Steam ID
    # --------------------------------------------------------

    steam_id = input(
        "Steam ID를 입력하세요: "
    ).strip()


    # --------------------------------------------------------
    # 데이터 로드
    # --------------------------------------------------------

    user_df, top500_df = load_data()


    # --------------------------------------------------------
    # 사용자 게임
    #
    # CSV에서 가져오는 것은 선호도 학습을 위해서입니다.
    # --------------------------------------------------------

    user_games = get_user_games(

        user_df,

        steam_id

    )


    print()

    print(
        f"학습에 사용되는 사용자 플레이 게임: "
        f"{len(user_games)}개"
    )


    # ========================================================
    # ★ Steam API로 현재 전체 라이브러리 조회
    # ========================================================

    owned_appids = get_owned_games_from_steam(
        steam_id
    )


    # --------------------------------------------------------
    # 플레이타임 가중치
    # --------------------------------------------------------

    weighted_games = calculate_weights(
        user_games
    )


    # --------------------------------------------------------
    # 사용자 선호도
    # --------------------------------------------------------

    user_preference = (
        build_user_preference(
            weighted_games
        )
    )


    print()
    print(
        "=== 사용자 장르 선호도 ==="
    )


    for genre, value in sorted(

        user_preference.items(),

        key=lambda x: x[1],

        reverse=True

    ):

        print(
            f"{genre:15s}: "
            f"{value:.4f}"
        )


    # --------------------------------------------------------
    # 게임 Feature
    # --------------------------------------------------------

    game_features = (
        create_game_features(
            top500_df
        )
    )


    # --------------------------------------------------------
    # 학습 데이터
    # --------------------------------------------------------

    X, y = create_training_data(

        user_games,

        game_features

    )


    # --------------------------------------------------------
    # 학습 데이터 확인
    # --------------------------------------------------------

    if len(X) < 10:

        raise ValueError(
            "학습 데이터가 너무 적습니다."
        )


    if len(
        np.unique(y)
    ) < 2:

        raise ValueError(
            "Positive와 Negative 데이터가 "
            "둘 다 필요합니다."
        )


    # --------------------------------------------------------
    # XGBoost 학습
    # --------------------------------------------------------

    model = train_xgboost(
        X,
        y
    )


    # ========================================================
    # 추천
    # ========================================================

    recommendations = recommend_games(

        model,

        game_features,

        user_preference,

        owned_appids,

        top_n=20

    )


    # --------------------------------------------------------
    # 결과 출력
    # --------------------------------------------------------

    print()
    print()
    print("=" * 70)

    print(
        "=== 개인 맞춤 추천 TOP 20 ==="
    )

    print("=" * 70)


    print()


    for _, row in recommendations.iterrows():

        print(
            f"{int(row['rank']):2d}. "
            f"{row['game_name']}"
        )

        print(
            f"    AppID: "
            f"{row['appid']}"
        )

        print(
            f"    XGBoost 좋아할 확률: "
            f"{row['xgboost_probability']:.2%}"
        )

        print(
            f"    장르 선호도: "
            f"{row['genre_preference_score']:.4f}"
        )

        print(
            f"    평점: "
            f"{row['rating']:.2f}"
        )

        print()


    # --------------------------------------------------------
    # CSV 저장
    # --------------------------------------------------------

    save_columns = [

        "rank",

        "appid",

        "game_name",

        "rating",

        "xgboost_probability",

        "genre_preference_score"

    ]


    recommendations[
        save_columns
    ].to_csv(

        OUTPUT_CSV,

        index=False,

        encoding="utf-8-sig"

    )


    print("=" * 70)

    print(
        "추천 결과 저장 완료"
    )

    print(
        OUTPUT_CSV
    )

    print("=" * 70)


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":

    main()

Steam XGBoost 개인 맞춤 추천 시스템


Steam ID를 입력하세요:  76561198056237344


CSV 파일 로딩
사용자 데이터: 4,500개
TOP500 데이터: 95개

학습에 사용되는 사용자 플레이 게임: 30개

Steam 라이브러리 실시간 조회
현재 Steam 라이브러리: 306개

=== 사용자 장르 선호도 ===
Action         : 0.3598
Simulation     : 0.1568
RPG            : 0.1375
Adventure      : 0.1316
Strategy       : 0.1065
Casual         : 0.0671
Horror         : 0.0204
Survival       : 0.0204
Shooter        : 0.0000
Sports         : 0.0000
Racing         : 0.0000
Platformer     : 0.0000
Puzzle         : 0.0000

XGBoost 학습
학습 데이터: 95개
Positive: 13개
Negative: 82개

=== XGBoost 평가 ===
Accuracy : 0.7368
Precision: 0.0000
Recall   : 0.0000
F1-score : 0.0000
ROC-AUC  : 0.2708
PR-AUC   : 0.1398

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.82      0.88      0.85        16
           1       0.00      0.00      0.00         3

    accuracy                           0.74        19
   macro avg       0.41      0.44      0.42        19
weighted avg       0.69      0.74      0.71        19


이미 보유한 게임 제외
추천 후보 게